# 컨텍스트 .mvna 파일 pandas 확인

컨텍스트에 포함된 두 파일 바이트를 pandas 테이블로 읽고, 사람이 보기 쉽게 표시한 뒤 동일성/차이를 검증합니다.

In [36]:
from pathlib import Path
import binascii
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
project_root = cwd if (cwd / "AGENTS.md").exists() else cwd.parent
print("CWD:", cwd)
print("PROJECT_ROOT:", project_root)

file1_path = project_root / "data/ACLR/ACLR31/Participant.mvna"
file2_path = project_root / "data/ACLD/ACLD36/Participant.mvna"
input_files = [file1_path, file2_path]

pd.options.display.max_columns = 20
pd.options.display.width = 140
pd.options.display.max_colwidth = 120

input_files

CWD: /Users/ryutt/Desktop/mini_ryutt/Walking/notebooks
PROJECT_ROOT: /Users/ryutt/Desktop/mini_ryutt/Walking


[PosixPath('/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLR/ACLR31/Participant.mvna'),
 PosixPath('/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLD/ACLD36/Participant.mvna')]

## 1) 필수 라이브러리 임포트 및 입력 파일 경로 정의

`pandas`, `pathlib`, `binascii`를 사용하고, VS Code 현재 작업 디렉터리를 출력해 경로 기준을 고정합니다.

## 2) .mvna 파일을 바이너리로 읽고 기본 메타데이터(DataFrame) 생성

각 파일의 크기와 헤더(hex)를 테이블로 확인합니다.

In [37]:
file_bytes = {}
meta_rows = []

for p in input_files:
    data = p.read_bytes()
    file_bytes[str(p)] = data
    meta_rows.append(
        {
            "path": str(p),
            "exists": p.exists(),
            "size_bytes": len(data),
            "header_hex_32": binascii.hexlify(data[:32]).decode("ascii"),
        }
    )

meta_df = pd.DataFrame(meta_rows)
meta_df

,path,exists,size_bytes,header_hex_32
0,/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLR/ACLR31/Participant.mvna,True,153,23735f66483e3e57595e2d020854724f4217546548324a225d6119edaad6a9c2
1,/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLD/ACLD36/Participant.mvna,True,186,23705f66483e3e57595e2d020854724544037a44492f454a6f29be0d27e1778a


## 3) 바이트 스트림을 판다스 테이블(Offset, Hex, ASCII)로 변환

고정 폭(16바이트)으로 나눠 `offset`, `hex`, `ascii` 컬럼을 만듭니다.

In [38]:
def bytes_to_table(data: bytes, width: int = 16) -> pd.DataFrame:
    rows = []
    for offset in range(0, len(data), width):
        chunk = data[offset : offset + width]
        rows.append(
            {
                "offset": offset,
                "hex": " ".join(f"{b:02X}" for b in chunk),
                "ascii": "".join(chr(b) if 32 <= b <= 126 else "." for b in chunk),
            }
        )
    return pd.DataFrame(rows)


tables = {str(p): bytes_to_table(file_bytes[str(p)], width=16) for p in input_files}
list(tables.keys())

['/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLR/ACLR31/Participant.mvna',
 '/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLD/ACLD36/Participant.mvna']

## 4) 두 파일의 DataFrame 미리보기 및 표시 옵션 조정

각 파일에 대해 `head()`, `tail()`, 그리고 앞쪽 offset 구간을 출력합니다.

In [39]:
for p in input_files:
    key = str(p)
    df = tables[key]
    print(f"\n===== {key} =====")
    print(f"rows={len(df)}")

    print("\n[head]")
    display(df.head(5))

    print("\n[tail]")
    display(df.tail(5))

    print("\n[offset 0~63]")
    display(df[(df["offset"] >= 0) & (df["offset"] < 64)])


===== /Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLR/ACLR31/Participant.mvna =====
rows=10

[head]


,offset,hex,ascii
0,0,23 73 5F 66 48 3E 3E 57 59 5E 2D 02 08 54 72 4F,#s_fH>>WY^-..TrO
1,16,42 17 54 65 48 32 4A 22 5D 61 19 ED AA D6 A9 C2,"B.TeH2J""]a......"
2,32,BB A1 0D 61 5F 19 29 08 55 71 0F 45 4A 44 33 54,...a_.).Uq.EJD3T
3,48,4D 08 4D 53 19 48 73 4E DA CD 39 DF 5E E0 04 73,M.MS.HsN..9.^..s
4,64,3E 49 03 30 20 35 5A 32 4D 14 3F 12 1E 3D 36 21,>I.0 5Z2M.?..=6!



[tail]


,offset,hex,ascii
5,80,4F 30 6B 71 48 52 3E 83 97 5A 88 4C 3F 2D 3D 2B,O0kqHR>..Z.L?-=+
6,96,3F 4B 2D 2C 0E 8A 1D 31 03 2D 21 3A 5D 4A 59 2B,"?K-,...1.-!:]JY+"
7,112,46 27 CF 74 71 A2 B1 D8 CD 12 0C 4C 29 5D 0E 2D,F'.tq......L)].-
8,128,33 5A 22 1E 3A 4C 37 5E 53 76 1F 34 00 4F 41 66,"3Z"".:L7^Sv.4.OAf"
9,144,6E 6E F5 FF 4B 04 51 93 16,nn..K.Q..



[offset 0~63]


...

,offset,hex,ascii
0,0,23 70 5F 66 48 3E 3E 57 59 5E 2D 02 08 54 72 45,#p_fH>>WY^-..TrE
1,16,44 03 7A 44 49 2F 45 4A 6F 29 BE 0D 27 E1 77 8A,D.zDI/EJo)..'.w.
2,32,1D 0B 53 55 4A 15 2F 23 4E 58 48 4A 0B 46 25 73,..SUJ./#NXHJ.F%s
3,48,5C 32 43 41 08 48 73 62 6C 15 5B 54 73 CE 04 73,\2CA.Hsbl.[Ts..s
4,64,3E 49 03 30 20 35 5A 32 4D 0F 3E 0D 33 0B 30 2A,>I.0 5Z2M.>.3.0*



[tail]


,offset,hex,ascii
7,112,2F 1B 5C 5C 6B 95 6D A8 C6 7C E3 E3 75 7E 00 3C,/.\\k.m..|..u~.<
8,128,03 5C 3D 74 32 4F 72 59 58 5B 1F 15 02 4E 52 0E,.\=t2OrYX[...NR.
9,144,5C 64 64 00 B4 FB AE 0C 75 A3 77 62 30 3C 52 07,\dd.....u.wb0<R.
10,160,15 04 5C 56 0A 3E 44 1C 33 37 39 56 36 04 14 22,"..\V.>D.379V6.."""
11,176,62 10 B4 0E FE 39 C1 80 86 06,b....9....



[offset 0~63]


,offset,hex,ascii
0,0,23 70 5F 66 48 3E 3E 57 59 5E 2D 02 08 54 72 45,#p_fH>>WY^-..TrE
1,16,44 03 7A 44 49 2F 45 4A 6F 29 BE 0D 27 E1 77 8A,D.zDI/EJo)..'.w.
2,32,1D 0B 53 55 4A 15 2F 23 4E 58 48 4A 0B 46 25 73,..SUJ./#NXHJ.F%s
3,48,5C 32 43 41 08 48 73 62 6C 15 5B 54 73 CE 04 73,\2CA.Hsbl.[Ts..s


## 5) 두 파일 일치 여부 검증과 바이트 단위 diff 테이블 생성

길이/전체 bytes 비교를 수행하고, 불일치가 있으면 `index`, `file1_byte`, `file2_byte`, `xor` 컬럼의 diff DataFrame을 생성합니다.

In [40]:
b1 = file_bytes[str(file1_path)]
b2 = file_bytes[str(file2_path)]

same_length = len(b1) == len(b2)
same_bytes = b1 == b2

print("same_length:", same_length)
print("same_bytes :", same_bytes)

min_len = min(len(b1), len(b2))
diff_rows = []

for i in range(min_len):
    x = b1[i]
    y = b2[i]
    if x != y:
        diff_rows.append(
            {
                "index": i,
                "file1_byte": x,
                "file2_byte": y,
                "xor": x ^ y,
            }
        )

if len(b1) != len(b2):
    longer, label = (b1, "file1_byte") if len(b1) > len(b2) else (b2, "file2_byte")
    for i in range(min_len, len(longer)):
        row = {"index": i, "file1_byte": None, "file2_byte": None, "xor": None}
        row[label] = longer[i]
        diff_rows.append(row)

diff_df = pd.DataFrame(diff_rows)

if diff_df.empty:
    print("diff 없음: 두 파일 바이트가 완전히 동일합니다.")
else:
    print(f"diff 개수: {len(diff_df)}")
    display(diff_df.head(50))

same_length: False
same_bytes : False
diff 개수: 158


,index,file1_byte,file2_byte,xor
0,1,115.0,112,3.0
1,15,79.0,69,10.0
2,16,66.0,68,6.0
3,17,23.0,3,20.0
4,18,84.0,122,46.0
5,19,101.0,68,33.0
6,20,72.0,73,1.0
7,21,50.0,47,29.0
8,22,74.0,69,15.0
9,23,34.0,74,104.0


## 6) 간단 실행 검증용 assert 셀 구성

파일 존재/비어있지 않음/테이블 생성/비교 로직을 `assert`로 점검합니다.

In [41]:
assert all(p.exists() for p in input_files), "입력 파일 중 일부가 존재하지 않습니다."
assert all(p.stat().st_size > 0 for p in input_files), "입력 파일 중 빈 파일이 있습니다."

assert isinstance(meta_df, pd.DataFrame) and not meta_df.empty, "메타데이터 DataFrame 생성 실패"
assert all(isinstance(tables[str(p)], pd.DataFrame) for p in input_files), "바이트 테이블 생성 실패"
assert isinstance(diff_df, pd.DataFrame), "diff DataFrame 생성 실패"

if same_bytes:
    assert diff_df.empty, "동일 파일인데 diff가 비어있지 않습니다."

print("모든 실행 검증(assert) 통과")

모든 실행 검증(assert) 통과


## 7) ACLR 폴더 전체 재귀 탐색 후 기준 파일과 diff 없는 파일 찾기

`data/ACLR` 하위의 모든 `.mvna` 파일을 재귀 탐색하고, 기준 파일(`file2_path`)과 바이트 단위로 완전히 동일한 파일만 추립니다.

In [42]:
aclr_root = project_root / "data/ACLR"
reference_path = file2_path
reference_bytes = reference_path.read_bytes()

all_mvna_paths = sorted(aclr_root.rglob("*.mvna"))
match_rows = []

for p in all_mvna_paths:
    data = p.read_bytes()
    same_size = len(data) == len(reference_bytes)
    same_bytes_full = same_size and (data == reference_bytes)

    if same_bytes_full:
        match_rows.append(
            {
                "path": str(p),
                "relative_path": str(p.relative_to(project_root)),
                "size_bytes": len(data),
            }
        )

matches_df = pd.DataFrame(match_rows)

print("reference:", reference_path)
print("scanned mvna files in ACLR:", len(all_mvna_paths))
print("no-diff matches:", len(matches_df))

if matches_df.empty:
    print("diff 없는 파일을 찾지 못했습니다.")
else:
    display(matches_df)

reference: /Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLD/ACLD36/Participant.mvna
scanned mvna files in ACLR: 22
no-diff matches: 1


,path,relative_path,size_bytes
0,/Users/ryutt/Desktop/mini_ryutt/Walking/data/ACLR/ACLR38/Participant.mvna,data/ACLR/ACLR38/Participant.mvna,186


In [43]:
assert aclr_root.exists(), "ACLR 폴더를 찾지 못했습니다."
assert len(all_mvna_paths) > 0, "ACLR 하위 mvna 파일이 없습니다."

if not matches_df.empty:
    # 매칭으로 나온 파일은 반드시 기준 파일과 완전 동일해야 함
    assert all((Path(p).read_bytes() == reference_bytes) for p in matches_df["path"]), "매칭 결과에 불일치 파일이 포함되어 있습니다."

print("ACLR 재귀 탐색 및 no-diff 매칭 검증 통과")

ACLR 재귀 탐색 및 no-diff 매칭 검증 통과
